In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn import preprocessing
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [2]:
#Einlesen der Daten und windzorising der obersten und unteresten 2% (gemäß Angaben im Paper)
fs1_wo_lag = pd.read_csv('data/feature_sets/feature_set_1_without_lag.csv', index_col=0)
fs1_wo_lag.clip(lower=fs1_wo_lag.quantile(0.02), upper=fs1_wo_lag.quantile(0.98), axis=1, inplace=True)

fs1_lag = pd.read_csv('data/feature_sets/feature_set_1_lag.csv', index_col=0)
fs1_lag.clip(lower=fs1_lag.quantile(0.02), upper=fs1_lag.quantile(0.98), axis=1, inplace=True)

fs2 = pd.read_csv('data/feature_sets/feature_set_2_calculated.csv')
fs2.clip(lower=fs2.quantile(0.02), upper=fs2.quantile(0.98), axis=1, inplace=True)

fs3 = pd.read_csv('data/feature_sets/feature_set_3_calculated.csv')
fs3.clip(lower=fs3.quantile(0.02), upper=fs3.quantile(0.98), axis=1, inplace=True)


In [3]:
fs1_wo_lag.reset_index(inplace=True, drop=True)
fs1_wo_lag

,YEAR,QUARTER,DNGDP2_mean,DNGDP3_mean,DNGDP4_mean,DNGDP5_mean,DNGDP6_mean,DNGDP2_median,DNGDP3_median,DNGDP4_median,...,RGDP2_mean,RGDP3_mean,RGDP4_mean,RGDP5_mean,RGDP6_mean,RGDP2_median,RGDP3_median,RGDP4_median,RGDP5_median,RGDP6_median
0,1989,1,8.4050,6.8171,6.3360,5.93780,5.98510,8.6009,6.3492,6.4930,...,4126.02161,4139.26720,4150.33031,4173.27397,4201.77099,4126.20000,4137.70000,4150.15000,4173.4000,4204.25500
1,1989,2,7.0945,6.3669,6.2514,6.25620,6.31950,6.4838,6.1388,5.6504,...,4126.02161,4139.26720,4150.33031,4173.27397,4201.77099,4126.20000,4137.70000,4150.15000,4173.4000,4204.25500
2,1989,3,5.9025,6.1218,6.6899,4.99410,6.62617,5.8207,5.7768,6.6341,...,4143.51330,4161.88670,4182.73330,4205.55330,4231.02670,4140.00000,4160.00000,4170.00000,4186.0000,4226.90000
3,1989,4,5.3374,5.5528,6.1549,6.20880,6.62617,5.1014,5.8167,6.8934,...,4171.50000,4187.25000,4205.81250,4229.00000,4258.06250,4169.50000,4187.50000,4204.00000,4228.5000,4253.00000
4,1990,1,5.4898,5.9550,6.4940,6.60336,6.62617,5.2726,5.5503,6.0454,...,4184.78570,4205.42860,4227.21430,4253.07140,4278.00000,4181.50000,4201.50000,4226.50000,4254.5000,4278.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131,2021,4,8.5961,7.4053,6.4573,5.97450,5.42350,8.6682,7.8000,6.8957,...,19696.05090,19893.88270,20079.37600,20224.63059,20381.62573,19686.15030,19874.39330,20070.30690,20222.3504,20378.45830
132,2022,1,5.8560,7.8401,6.4712,6.01840,5.81310,6.0000,7.7273,5.9223,...,19773.28239,19944.76603,20108.13713,20224.63059,20381.62573,19780.29704,19942.02031,20089.10027,20230.0007,20378.75932
133,2022,2,7.2323,7.0835,6.2193,5.77550,5.23650,7.2941,6.3936,5.2917,...,19773.28239,19944.76603,20100.73550,20213.04540,20326.94300,19780.29704,19942.02031,20087.89520,20192.5609,20306.80440
134,2022,3,6.0054,5.6585,4.9230,4.66500,4.29930,6.6900,5.0153,5.2949,...,19743.72960,19816.75310,19877.17070,19947.08170,20023.57050,19750.22720,19811.02800,19865.89880,19937.5378,20011.58410


In [4]:
fs1_lag.reset_index(inplace=True, drop=True)
fs1_lag = fs1_lag.shift(1)
fs1_lag

,Year,Quarter,First,Second
0,NaN,NaN,NaN,NaN
1,1989.0,1.0,9.70290,8.69440
2,1989.0,2.0,6.55450,7.28650
3,1989.0,3.0,5.61260,6.07000
4,1989.0,4.0,4.30960,4.35650
...,...,...,...,...
131,2021.0,3.0,7.82720,8.07870
132,2021.0,4.0,11.40714,11.62994
133,2022.0,1.0,6.48280,6.51080
134,2022.0,2.0,7.84980,8.39070


In [5]:
fs2 = fs2.shift(1) # Shift um ein Quartal -> Daten die Ende Januar veröffentlicht werden sind Daten für das letzte Quartal aus dem Vorjahr
fs2

,fyearq,fqtr,prccq_sd_dif,prccq_sd_yoy,prccq_vwmean_dif,prccq_mean_dif,prccq_vwmean_yoy,prccq_mean_yoy,prchq_sd_dif,prchq_sd_yoy,prchq_vwmean_dif,prchq_mean_dif,prchq_vwmean_yoy,prchq_mean_yoy,prclq_sd_dif,prclq_sd_yoy,prclq_vwmean_dif,prclq_mean_dif,prclq_vwmean_yoy,prclq_mean_yoy
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1989.0,1.0,NaN,NaN,0.000000,NaN,0.000000,NaN,NaN,NaN,0.000000,NaN,0.000000,NaN,NaN,NaN,0.000000,NaN,0.000000,NaN
2,1989.0,2.0,19.519145,NaN,1.688910,0.380724,0.000000,NaN,25.865517,NaN,4.153058,0.983916,0.000000,NaN,5.052074,NaN,1.301944,0.421802,0.000000,NaN
3,1989.0,3.0,31.304590,NaN,3.915837,0.206428,0.000000,NaN,22.710473,NaN,4.889874,0.891739,0.000000,NaN,22.261509,NaN,4.701553,0.786088,0.000000,NaN
4,1989.0,4.0,21.423680,16.584560,-5.525103,-0.847429,-0.007820,-1.014078,6.178982,10.375290,-3.597754,-0.150304,-0.005296,-1.419571,20.290963,10.542783,-1.678948,-0.199917,-0.003764,-1.419571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,2019.0,3.0,44.808873,87.389800,0.867011,0.445466,-0.892867,-1.141244,79.561656,72.340179,4.991220,1.920267,5.784017,0.280597,22.450220,79.778774,5.465174,0.555905,5.454136,-1.171391
124,2019.0,4.0,24.039349,63.296738,12.848948,0.932771,36.420574,3.839770,32.380153,160.945131,2.001552,0.221771,10.546304,-0.271993,36.368314,68.835080,3.976858,0.705453,28.703634,4.003325
125,2020.0,1.0,121.280550,98.762191,-9.755619,-5.497223,13.592352,-5.604409,23.514230,72.616785,23.857657,1.863892,42.440450,4.566513,154.441795,130.199390,-20.365622,-7.812768,4.916306,-10.302365
126,2020.0,2.0,38.659753,113.310113,42.010874,2.860910,49.137211,-2.725603,165.559988,139.634977,16.700349,-5.265100,50.960139,-1.963619,42.274652,153.742922,16.872664,2.318336,8.924623,-10.149343


In [6]:
fs3.drop(['pm_sd_dif', 'pm_sd_yoy', 'dep_vwmean_yoy', 'pm_mean_yoy', 'pm_vwmean_yoy', 'pm_mean_dif', 'pm_vwmean_dif', 'dep_mean_yoy'], inplace=True, axis=1) #Spalten mit fehlerhaften Werten werden entfernt -> TODO: Berechnung der Werte kontrollieren
fs3 = fs3.shift(1) # Shift um ein Quartal -> Daten die Ende Januar veröffentlicht werden sind Daten für das letzte Quartal aus dem Vorjahr
fs3

,fyearq,fqtr,apq_sd_dif,apq_sd_yoy,apq_vwmean_dif,apq_mean_dif,apq_vwmean_yoy,apq_mean_yoy,capsq_sd_dif,capsq_sd_yoy,...,xoprq_vwmean_dif,xoprq_mean_dif,xoprq_vwmean_yoy,xoprq_mean_yoy,xsgaq_sd_dif,xsgaq_sd_yoy,xsgaq_vwmean_dif,xsgaq_mean_dif,xsgaq_vwmean_yoy,xsgaq_mean_yoy
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1989.0,1.0,NaN,NaN,0.000000,NaN,0.000000,NaN,NaN,NaN,...,0.000000,NaN,0.000000,NaN,NaN,NaN,0.000000,NaN,0.000000,NaN
2,1989.0,2.0,901.335416,NaN,-52.320109,-22.169756,0.000000,NaN,381.522611,NaN,...,101.678560,3.707840,0.000000,NaN,30.035355,NaN,18.166209,-0.665318,0.000000,NaN
3,1989.0,3.0,922.499860,NaN,53.374532,-21.289743,0.000000,NaN,370.211646,NaN,...,-56.031159,-0.780296,0.000000,NaN,29.399768,NaN,6.598938,-0.922319,0.000000,NaN
4,1989.0,4.0,927.719642,985.683159,179.214321,-15.961623,0.031354,3.924847,386.535534,176.994649,...,264.831045,11.801106,0.019645,3.613921,36.100695,30.181055,49.937089,0.710723,0.000377,-3.415303
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,2019.0,3.0,642.193692,1704.077084,677.825453,32.466333,2000.338607,104.172152,586.478420,1169.851043,...,33.185912,2.204269,660.559699,10.700776,73.350313,157.131943,84.206364,0.559558,313.089029,3.610338
124,2019.0,4.0,2147.021282,2594.816617,1199.242186,33.340228,2493.044653,122.991803,677.638890,1299.742728,...,1046.478597,25.902268,688.944749,11.019481,77.683109,180.794929,251.875918,6.325520,354.035720,4.039635
125,2020.0,1.0,3131.944606,5955.683507,2857.731927,69.371778,6346.823319,251.802537,425.480999,1337.167493,...,-77.940904,-24.400975,1177.031448,16.998803,97.141856,131.400266,-206.105281,-6.266933,360.176911,2.714064
126,2020.0,2.0,2311.907486,7426.060695,1748.086343,69.371778,7987.846236,296.993145,408.865624,1151.631845,...,-671.976033,-36.811097,601.388469,-34.693335,75.553160,133.865696,106.792807,-2.655511,340.069755,-2.554501


In [8]:
#Kontrolle, ob noch nulls in den Daten sind
df = fs3.iloc[4:124,]
df1 = df[df.isna().any(axis=1)]
df1

,fyearq,fqtr,apq_sd_dif,apq_sd_yoy,apq_vwmean_dif,apq_mean_dif,apq_vwmean_yoy,apq_mean_yoy,capsq_sd_dif,capsq_sd_yoy,...,xoprq_vwmean_dif,xoprq_mean_dif,xoprq_vwmean_yoy,xoprq_mean_yoy,xsgaq_sd_dif,xsgaq_sd_yoy,xsgaq_vwmean_dif,xsgaq_mean_dif,xsgaq_vwmean_yoy,xsgaq_mean_yoy


In [9]:
#Kontrolle, ob noch nulls in den Daten sind
df2 = df.isna().any()
df2

fyearq              False
fqtr                False
apq_sd_dif          False
apq_sd_yoy          False
apq_vwmean_dif      False
                    ...  
xsgaq_sd_yoy        False
xsgaq_vwmean_dif    False
xsgaq_mean_dif      False
xsgaq_vwmean_yoy    False
xsgaq_mean_yoy      False
Length: 277, dtype: bool

In [10]:
#Kombinieren der eingelesenen Daten zu den im Paper angegeben Feature Set Kombinationen
frames = [fs1_wo_lag[fs1_wo_lag.columns[2:]],
          fs1_lag['First']
          ]
survey_estimates = pd.concat(frames, axis=1)

frames = [fs1_wo_lag[fs1_wo_lag.columns[2:]],
          fs1_lag[fs1_lag.columns[2:]],
          fs2[fs2.columns[2:]]
          ]
s_e_market = pd.concat(frames, axis=1)

frames = [fs1_wo_lag[fs1_wo_lag.columns[2:]],
          fs1_lag[fs1_lag.columns[2:]],
          fs3[fs3.columns[2:]]
          ]
s_e_accounting = pd.concat(frames, axis=1)

frames = [fs1_wo_lag[fs1_wo_lag.columns[2:]],
          fs1_lag[fs1_lag.columns[2:]],
          fs2[fs2.columns[2:]],
          fs3[fs3.columns[2:]]
          ]
all_data = pd.concat(frames, axis=1)

#Einlesen des Targets (BEA final estimates)
target = pd.read_csv('data/target.csv')
target = target.iloc[0:124,]
target.clip(lower=target.quantile(0.02), upper=target.quantile(0.98), axis=1, inplace=True) # windzorising der obersten und unteresten 2%
#Shifts für die verschieden Quartale zum Vorhersagen
target_curr_quarter = target['Third']
target_Q1 = target['Third'].shift(-1)
target_Q1 = target_Q1.iloc[0:123]
target_Q2 = target['Third'].shift(-2)
target_Q2 = target_Q2.iloc[0:122]
target_Q3 = target['Third'].shift(-3)
target_Q3 = target_Q3.iloc[0:121]
target_Q4 = target['Third'].shift(-4)
target_Q4 = target_Q4.iloc[0:120]

In [11]:
survey_estimates = preprocessing.StandardScaler().fit_transform(survey_estimates)
s_e_market = preprocessing.StandardScaler().fit_transform(s_e_market)
s_e_accounting = preprocessing.StandardScaler().fit_transform(s_e_accounting)
all_data = preprocessing.StandardScaler().fit_transform(all_data)

In [12]:
def grid_search(X, y, expanding=True):
    tested_acc = [0, 0, 0]
    results = []
    #Werte aus dem original Paper
    alpha_values = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]
    lambda_values = [0.01, 0.03, 0.1, 0.3, 1, 3, 10]

    best_acc = [0, 0, 0]
    for a in alpha_values:
        for l in lambda_values:

            if expanding:
                curr_mse = walk_forward_expanding(X, y, lam=l, l_ratio=a) # eigene Implementierung einer Walkforward validation zur Findung der besten Parameter
            else:
                curr_mse = walk_forward_rolling(X, y, lam=l, l_ratio=a)

            tested_acc[0] = curr_mse
            tested_acc[1] = a
            tested_acc[2] = l

            results.append(tested_acc.copy())
            if best_acc[0] == 0:
                best_acc[0] = curr_mse
                best_acc[1] = a
                best_acc[2] = l

            elif curr_mse < best_acc[0]:
                best_acc[0] = curr_mse
                best_acc[1] = a
                best_acc[2] = l

    results_df = pd.DataFrame(results)
    return results_df, best_acc[1], best_acc[2]

In [13]:
def grid_search_nn(X, y, expanding=True):
    tested_acc = [0, 0, 0]
    results = []
    #Werte aus dem original Paper
    hiddenlayer_size = [1, 2, 3, 4, 5]
    alphas = [0.0001, 0.001, 0.01, 0.1, 1.0]


    best_acc = [0, 0, 0]
    for h in hiddenlayer_size:
        for a in alphas:

            if expanding:
                curr_mse = walk_forward_expanding_nn(X, y, hiddenlayer_size=h, alpha=a) # eigene Implementierung einer Walkforward validation zur Findung der besten Parameter
            else:
                curr_mse = walk_forward_rolling_nn(X, y, hiddenlayer_size=h, alpha=a)

            tested_acc[0] = curr_mse
            tested_acc[1] = h
            tested_acc[2] = a

            results.append(tested_acc.copy())
            if best_acc[0] == 0:
                best_acc[0] = curr_mse
                best_acc[1] = h
                best_acc[2] = a

            elif curr_mse < best_acc[0]:
                best_acc[0] = curr_mse
                best_acc[1] = h
                best_acc[2] = a

    results_df = pd.DataFrame(results)
    return results_df, best_acc[1], best_acc[2]

In [14]:
def grid_search_rf(X, y, expanding=True):
    tested_acc = [0, 0, 0, 0]
    results = []
    #Werte aus dem original Paper
    max_depth = [2, 4, 6, 8, 10, 12]
    max_features = [1, 2, 3]
    estimators = [100, 200, 300, 400, 500]


    best_acc = [0, 0, 0, 0]
    for d in max_depth:
        for f in max_features:
            for e in estimators:

                if expanding:
                    curr_mse = walk_forward_expanding_rf(X, y, max_depth=d, max_features=f, estimators=e) # eigene Implementierung einer Walkforward validation zur Findung der besten Parameter
                else:
                    curr_mse = walk_forward_rolling_rf(X, y, max_depth=d, max_features=f, estimators=e)

                tested_acc[0] = curr_mse
                tested_acc[1] = d
                tested_acc[2] = f
                tested_acc[3] = e

                results.append(tested_acc.copy())
                if best_acc[0] == 0:
                    best_acc[0] = curr_mse
                    best_acc[1] = d
                    best_acc[2] = f
                    best_acc[3] = e

                elif curr_mse < best_acc[0]:
                    best_acc[0] = curr_mse
                    best_acc[1] = d
                    best_acc[2] = f
                    best_acc[3] = e

    results_df = pd.DataFrame(results)
    return results_df, best_acc[1], best_acc[2], best_acc[3]

In [15]:
def walk_forward_rolling(X, y, lam, l_ratio):
    test_mse = []
    for endIdx in range(24, 83, 20):

        X_train = X[endIdx-20:endIdx,]                 #X.iloc[4:endIdx,]
        X_test = X[endIdx:endIdx+20,]          #X.iloc[endIdx:endIdx+20,]
        y_train = y[endIdx-20:endIdx,]                 #y.iloc[4:endIdx,]
        y_test = y[endIdx:endIdx+20,]          #y.iloc[endIdx:endIdx+20,]

        en_en = ElasticNet(random_state=42, alpha=lam, l1_ratio=l_ratio)
        en_en.fit(X_train, y_train)

        en_predictions_en = en_en.predict(X_test)

        en_mse_en = mean_squared_error(y_test, en_predictions_en)
        test_mse.append(en_mse_en)
    return np.mean(test_mse)

def walk_forward_expanding(X, y, lam, l_ratio):
    test_mse = []
    for endIdx in range(24, 83, 20): #Trainingsdaten wachsen Schritt für Schritt an

        X_train = X[4:endIdx,]                 #X.iloc[4:endIdx,]
        X_test = X[endIdx:endIdx+20,]          #X.iloc[endIdx:endIdx+20,]
        y_train = y[4:endIdx,]                 #y.iloc[4:endIdx,]
        y_test = y[endIdx:endIdx+20,]          #y.iloc[endIdx:endIdx+20,]

        en_en = ElasticNet(random_state=42, alpha=lam, l1_ratio=l_ratio)
        en_en.fit(X_train, y_train)

        en_predictions_en = en_en.predict(X_test)

        en_mse_en = mean_squared_error(y_test, en_predictions_en)
        test_mse.append(en_mse_en)
    return np.mean(test_mse)

In [16]:
def walk_forward_rolling_nn(X, y, hiddenlayer_size, alpha):
    test_mse = []
    for endIdx in range(24, 83, 20):

        X_train = X[endIdx-20:endIdx,]                 #X.iloc[4:endIdx,]
        X_test = X[endIdx:endIdx+20,]          #X.iloc[endIdx:endIdx+20,]
        y_train = y[endIdx-20:endIdx,]                 #y.iloc[4:endIdx,]
        y_test = y[endIdx:endIdx+20,]          #y.iloc[endIdx:endIdx+20,]

        nn = MLPRegressor(random_state=42, alpha=alpha, hidden_layer_sizes=hiddenlayer_size, max_iter=500)
        nn.fit(X_train, y_train)

        nn_prediction = nn.predict(X_test)

        nn_mse = mean_squared_error(y_test, nn_prediction)
        test_mse.append(nn_mse)
    return np.mean(test_mse)

def walk_forward_expanding_nn(X, y, hiddenlayer_size, alpha):
    test_mse = []
    for endIdx in range(24, 83, 20):

        X_train = X[4:endIdx,]                 #X.iloc[4:endIdx,]
        X_test = X[endIdx:endIdx+20,]          #X.iloc[endIdx:endIdx+20,]
        y_train = y[4:endIdx,]                 #y.iloc[4:endIdx,]
        y_test = y[endIdx:endIdx+20,]          #y.iloc[endIdx:endIdx+20,]

        nn = MLPRegressor(random_state=42, alpha=alpha, hidden_layer_sizes=hiddenlayer_size, max_iter=500)
        nn.fit(X_train, y_train)

        nn_prediction = nn.predict(X_test)

        nn_mse = mean_squared_error(y_test, nn_prediction)
        test_mse.append(nn_mse)
    return np.mean(test_mse)

In [17]:
def walk_forward_rolling_rf(X, y, max_depth, max_features, estimators):
    test_mse = []
    for endIdx in range(24, 83, 20):

        X_train = X[endIdx-20:endIdx,]                 #X.iloc[4:endIdx,]
        X_test = X[endIdx:endIdx+20,]          #X.iloc[endIdx:endIdx+20,]
        y_train = y[endIdx-20:endIdx,]                 #y.iloc[4:endIdx,]
        y_test = y[endIdx:endIdx+20,]          #y.iloc[endIdx:endIdx+20,]

        rf = RandomForestRegressor(random_state=42, max_depth=max_depth, max_features=max_features, n_estimators=estimators)
        rf.fit(X_train, y_train)

        prediction = rf.predict(X_test)
        rf_mse= mean_squared_error(y_test, prediction)

        test_mse.append(rf_mse)

    return np.mean(test_mse)

def walk_forward_expanding_rf(X, y, max_depth, max_features, estimators):
    test_mse = []
    for endIdx in range(24, 83, 20):

        X_train = X[4:endIdx,]                 #X.iloc[4:endIdx,]
        X_test = X[endIdx:endIdx+20,]          #X.iloc[endIdx:endIdx+20,]
        y_train = y[4:endIdx,]                 #y.iloc[4:endIdx,]
        y_test = y[endIdx:endIdx+20,]          #y.iloc[endIdx:endIdx+20,]

        rf = RandomForestRegressor(random_state=42, max_depth=max_depth, max_features=max_features, n_estimators=estimators)
        rf.fit(X_train, y_train)

        prediction = rf.predict(X_test)
        rf_mse= mean_squared_error(y_test, prediction)

        test_mse.append(rf_mse)
    return np.mean(test_mse)

In [18]:
def weighted_mean(values, last_idx):
    sum = 0.0
    total_weight = 0.0
    current_weight = 1.0

    n = len(values)
    for i in range(n):
        sum += values[last_idx - i] * current_weight
        total_weight += current_weight
        current_weight *= 0.9

    return sum / total_weight

In [19]:
target.iloc[4:83,]

,Unnamed: 0,Year,Quarter,Third
4,98.0,1990,1,7.142200
5,99.0,1990,2,5.149200
6,100.0,1990,3,5.343300
7,101.0,1990,4,0.924400
8,102.0,1991,1,2.218200
...,...,...,...,...
78,172.0,2008,3,3.351700
79,173.0,2008,4,-1.271104
80,174.0,2009,1,-1.271104
81,175.0,2009,2,-0.754000


In [20]:
development_samples = {
    'FS1' : survey_estimates,
    'FS1+2' : s_e_market,
    'FS1+3' : s_e_accounting,
    'FS1+2+3' : all_data
}
targets = {
    'Current Q' : target_curr_quarter,
    'Current Q+1' : target_Q1,
    'Current Q+2' : target_Q2,
    'Current Q+3' : target_Q3,
    'Current Q+4' : target_Q4,
}
real_params = { #optimale Parameter für ElasticNet laut Paper
    'Current Q'   : {'FS1'      : (0.3, 0.3),
                     'FS1+2'    : (0.6, 1.0),
                     'FS1+3'    : (0.8, 1.0),
                     'FS1+2+3'  : (0.8, 1.0)},
    'Current Q+1' : {'FS1'      : (0.1, 3.0),
                     'FS1+2'    : (0.5, 3.0),
                     'FS1+3'    : (0.9, 1.0),
                     'FS1+2+3'  : (1.0, 0.3)},
    'Current Q+2' : {'FS1'      : (0.4, 3.0),
                     'FS1+2'    : (0.4, 3.0),
                     'FS1+3'    : (0.1, 3.0),
                     'FS1+2+3'  : (0.1, 3.0)},
    'Current Q+3' : {'FS1'      : (0.8, 1.0),
                     'FS1+2'    : (0.9, 1.0),
                     'FS1+3'    : (0.5, 1.0),
                     'FS1+2+3'  : (0.1, 3.0)},
    'Current Q+4' : {'FS1'      : (0.7, 1.0),
                     'FS1+2'    : (0.3, 0.1),
                     'FS1+3'    : (0.2, 1.0),
                     'FS1+2+3'  : (0.2, 1.0)},
}

real_params_rf = { #optimale Parameter für RandomForest laut Paper
    'Current Q'   : {'FS1'      : (12, 1, 300),
                     'FS1+2'    : (10, 1, 100),
                     'FS1+3'    : (6, 1, 300),
                     'FS1+2+3'  : (4, 2, 300)},
    'Current Q+1' : {'FS1'      : (6, 1, 100),
                     'FS1+2'    : (10, 1, 100),
                     'FS1+3'    : (4, 2, 100),
                     'FS1+2+3'  : (6, 3, 500)},
    'Current Q+2' : {'FS1'      : (2, 1, 100),
                     'FS1+2'    : (8, 1, 100),
                     'FS1+3'    : (8, 2, 200),
                     'FS1+2+3'  : (8, 2, 200)},
    'Current Q+3' : {'FS1'      : (12, 1, 300),
                     'FS1+2'    : (4, 2, 100),
                     'FS1+3'    : (4, 3, 200),
                     'FS1+2+3'  : (4, 2, 300)},
    'Current Q+4' : {'FS1'      : (12, 1, 200),
                     'FS1+2'    : (2, 1, 500),
                     'FS1+3'    : (10, 1, 200),
                     'FS1+2+3'  : (8, 1, 200)},
}

In [21]:
all_data[4:103,]   # all_data.iloc[4:103,]

array([[ 0.50653803,  0.91941886,  1.79111249, ...,  0.09955691,
        -1.10843141, -2.32685998],
       [ 1.13225973,  1.04603091,  1.09955684, ..., -1.07759943,
        -0.75466273, -2.32685998],
       [ 0.44220564,  0.18198178,  0.44545891, ...,  0.03358907,
        -0.45749701, -1.7897437 ],
       ...,
       [-0.66253236, -0.60957322, -0.36068055, ...,  1.06847775,
        -1.55810415, -2.32685998],
       [ 0.11164869, -0.16220453, -0.21183213, ..., -1.3223427 ,
        -0.82935408,  0.21641342],
       [-0.61110404, -0.09329376, -0.40403327, ...,  0.31972426,
        -0.6196399 ,  0.01404778]])

In [22]:
start = datetime.now()
# Berechnung der MSEs für die verschiedenen Feature Sets und Zeitabschnitte für unsere selbst gefunden Parameter, sowie die optimalen aus dem Paper
list_of_mse_paper = []
list_of_mse_own = []
list_of_mse_rf = []
list_of_mse_mean = []
list_of_mse_weighted_mean = []
list_of_coefs = []
list_of_mse_nn = []
list_of_mse_rf_own = []

for target_key, target_value in targets.items():
    print(f"\033[1m{target_key}: \033[0m")
    for feature_key, feature_value in development_samples.items():
        # Aufteilung der Daten gemäß Paper -> 99 Quartale im Trainingset und 21 im Testset
        X_train = feature_value[4:103,]                        #feature_value.iloc[4:103,]
        X_test = feature_value[103:len(target_value),]         #feature_value.iloc[103:len(target_value),]
        y_train = target_value[4:103,]                         #target_value.iloc[4:103,]
        y_test = target_value[103:,]                           #target_value.iloc[103:,]
        results_df, best_lambda, best_alpha = grid_search(feature_value, target_value, expanding=False) #finden der eigenen besten Parameter
        print(f"{feature_key}: \n"
              f"Bestes lambda (Regularisierung-Stärke): {best_alpha}\n"
              f"Beste l1-ratio (bestes alpha):          {best_lambda}")
        en_best = ElasticNet(random_state=42, alpha=best_alpha, l1_ratio=best_lambda)
        en_best.fit(X_train, y_train)
        pred_own = en_best.predict(X_test)
        en_mse_own = mean_squared_error(y_test, pred_own)
        list_of_coefs.append(en_best.coef_)

        nn_results_df, best_hidden_layer_number, best_alpha = grid_search_nn(feature_value, target_value, expanding=False)
        print(f"Beste Größe Hidden Layer:               {best_hidden_layer_number}\n"
              f"Bestes alpha:                           {best_alpha}")
        nn_best = MLPRegressor(random_state=42, hidden_layer_sizes=best_hidden_layer_number, alpha=best_alpha, max_iter=500)
        nn_best.fit(X_train, y_train)
        nn_pred = nn_best.predict(X_test)
        nn_mse_best = mean_squared_error(y_test, nn_pred)

        rf_results_df, best_depth, best_features, best_estimators = grid_search_rf(feature_value, target_value, expanding=False)
        print(f"{feature_key}: \n"
              f"Beste Max Depth:                        {best_depth}\n"
              f"Beste Max Features:                     {best_features}\n"
              f"Beste Max Estimators:                   {best_estimators}")
        rf_best = RandomForestRegressor(random_state=42, max_depth=best_depth, max_features=best_features, n_estimators=best_estimators)
        rf_best.fit(X_train, y_train)
        rf_pred = rf_best.predict(X_test)
        rf_mse_own = mean_squared_error(y_test, rf_pred)

        weighted_mean_pred = weighted_mean(y_train, last_idx=102)
        weighted_mean_pred = np.full_like(y_test, weighted_mean_pred)
        weighted_mean_mse_pred = mean_squared_error(y_test, weighted_mean_pred)

        print(f"MSE (eigne-Params):     {en_mse_own}\n"                 # Prediction mit eigenen Parametern
              f"MSE (Weighted Mean):    {weighted_mean_mse_pred}\n"     # Prediction mit dem gewichteten Mittelwert der vergangenen Quartale
              f"MSE (RF-eigene-Params): {rf_mse_own}\n"               # Prediction mit eigenen Parametern (RandomForest)
              f"MSE (NN):               {nn_mse_best}\n")

        list_of_mse_own.append(en_mse_own)
        list_of_mse_weighted_mean.append(weighted_mean_mse_pred)
        list_of_mse_nn.append(nn_mse_best)
        list_of_mse_rf_own.append(rf_mse_own)

print(datetime.now() - start)

Current Q: 
FS1: 
Bestes lambda (Regularisierung-Stärke): 1
Beste l1-ratio (bestes alpha):          0.1
Beste Größe Hidden Layer:               5
Bestes alpha:                           0.1
FS1: 
Beste Max Depth:                        2
Beste Max Features:                     1
Beste Max Estimators:                   100
MSE (eigne-Params):     2.2619790996293694
MSE (Weighted Mean):    2.5878126720598154
MSE (RF-eigene-Params): 2.4158066911134233
MSE (NN):               2.7495785391862944

FS1+2: 
Bestes lambda (Regularisierung-Stärke): 1
Beste l1-ratio (bestes alpha):          0.7
Beste Größe Hidden Layer:               3
Bestes alpha:                           1.0
FS1+2: 
Beste Max Depth:                        2
Beste Max Features:                     3
Beste Max Estimators:                   200
MSE (eigne-Params):     2.609460371012549
MSE (Weighted Mean):    2.5878126720598154
MSE (RF-eigene-Params): 2.1961858451559344
MSE (NN):               12.01772183106281

FS1+3: 
Bestes l

In [23]:
list_of_mse_nn

[2.7495785391862944,
 12.01772183106281,
 23.424667504407648,
 9.161889341052763,
 8.6882217298226,
 12.245617827643844,
 15.507292681245167,
 8.184016689311436,
 8.985918583411355,
 13.267155703468495,
 12.9751612269878,
 6.398563442984851,
 4.916567234349062,
 12.044645380449204,
 16.14647089247592,
 7.1876342653909795,
 7.266889145697473,
 11.554234694004059,
 13.655817229107146,
 8.986497417508883]

In [24]:
start = datetime.now()
# Berechnung der MSEs für die Verschiedenen Feature Sets und Zeitabschnitte für unsere selbst gefunden Parameter, sowie die optimalen aus dem Paper
list_of_mse_paper = []
list_of_mse_own = []
list_of_mse_rf = []
list_of_mse_mean = []
list_of_mse_weighted_mean = []
list_of_coefs = []
list_of_mse_nn = []
list_of_mse_rf_own = []

for target_key, target_value in targets.items():
    print(f"\033[1m{target_key}: \033[0m")
    for feature_key, feature_value in development_samples.items():
        # Aufteilung der Daten gemäß Paper -> 99 Quartale im Trainingset und 21 im Testset
        X_train = feature_value[4:103,]                        #feature_value.iloc[4:103,]
        X_test = feature_value[103:len(target_value),]         #feature_value.iloc[103:len(target_value),]
        y_train = target_value[4:103,]                         #target_value.iloc[4:103,]
        y_test = target_value[103:,]                           #target_value.iloc[103:,]
        results_df, best_lambda, best_alpha = grid_search(feature_value, target_value) #finden der eigenen besten Parameter
        print(f"{feature_key}: \n"
              f"Bestes lambda (Regularisierung-Stärke): {best_alpha}\n"
              f"Beste l1-ratio (bestes alpha):          {best_lambda}")
        en_best = ElasticNet(random_state=42, alpha=best_alpha, l1_ratio=best_lambda)
        en_best.fit(X_train, y_train)
        pred_own = en_best.predict(X_test)
        en_mse_own = mean_squared_error(y_test, pred_own)
        list_of_coefs.append(en_best.coef_)

        nn_results_df, best_hidden_layer_number, best_alpha = grid_search_nn(feature_value, target_value)
        print(f"Beste Größe Hidden Layer:               {best_hidden_layer_number}\n"
              f"Bestes alpha:                           {best_alpha}")
        nn_best = MLPRegressor(random_state=42, hidden_layer_sizes=best_hidden_layer_number, alpha=best_alpha, max_iter=500)
        nn_best.fit(X_train, y_train)
        nn_pred = nn_best.predict(X_test)
        nn_mse_best = mean_squared_error(y_test, nn_pred)

        rf_results_df, best_depth, best_features, best_estimators = grid_search_rf(feature_value, target_value)
        print(f"Beste Max Depth:                        {best_depth}\n"
              f"Beste Max Features:                     {best_features}\n"
              f"Beste Max Estimators:                   {best_estimators}")
        rf_best = RandomForestRegressor(random_state=42, max_depth=best_depth, max_features=best_features, n_estimators=best_estimators)
        rf_best.fit(X_train, y_train)
        rf_pred = rf_best.predict(X_test)
        rf_mse_own = mean_squared_error(y_test, rf_pred)

        paper_alpha, paper_lambda = real_params[target_key][feature_key] #Nutzung der besten Parameter für ElasticNet aus dem Paper
        en_paper_best = ElasticNet(random_state=42, alpha=paper_alpha, l1_ratio=paper_lambda)
        en_paper_best.fit(X_train, y_train)
        pred_paper = en_paper_best.predict(X_test)
        en_mse_paper = mean_squared_error(y_test, pred_paper)

        paper_depth, paper_features, paper_estimators = real_params_rf[target_key][feature_key] #Nutzung der besten Parameter für RandomForest aus dem Paper
        rf_paper_best = RandomForestRegressor(random_state=42, max_depth=paper_depth, max_features=paper_features, n_estimators=paper_estimators)
        rf_paper_best.fit(X_train, y_train)
        pred_paper = rf_paper_best.predict(X_test)
        rf_mse_paper = mean_squared_error(y_test, pred_paper)

        mean_pred = np.mean(y_train)
        mean_pred = np.full_like(y_test, mean_pred)
        mean_mse_pred = mean_squared_error(y_test, mean_pred)

        weighted_mean_pred = weighted_mean(y_train, last_idx=102)
        weighted_mean_pred = np.full_like(y_test, weighted_mean_pred)
        weighted_mean_mse_pred = mean_squared_error(y_test, weighted_mean_pred)

        print(f"MSE (eigne-Params):     {en_mse_own}\n"                 # Prediction mit eigenen Parametern
              f"MSE (Paper-Params):     {en_mse_paper}\n"               # Prediction mit Parametern aus dem Paper
              f"MSE (RF-Paper-Params):  {rf_mse_paper}\n"               # Prediction mit Parametern aus dem Paper (RandomForest)
              f"MSE (RF-eigene-Params): {rf_mse_own}\n"                 # Prediction mit eigenen Parametern (RandomForest)
              f"MSE (Mean-Pred):        {mean_mse_pred}\n"              # Prediction mit dem Mittelwert der vergangenen Quartale
              f"MSE (Weighted Mean):    {weighted_mean_mse_pred}\n"     # Prediction mit dem gewichteten Mittelwert der vergangenen Quartale
              f"MSE (NN):               {nn_mse_best}\n")

        list_of_mse_paper.append(en_mse_paper)
        list_of_mse_own.append(en_mse_own)
        list_of_mse_rf.append(rf_mse_paper)
        list_of_mse_mean.append(mean_mse_pred)
        list_of_mse_weighted_mean.append(weighted_mean_mse_pred)
        list_of_mse_nn.append(nn_mse_best)
        list_of_mse_rf_own.append(rf_mse_own)

print(datetime.now() - start)

Current Q: 
FS1: 
Bestes lambda (Regularisierung-Stärke): 0.1
Beste l1-ratio (bestes alpha):          1
Beste Größe Hidden Layer:               5
Bestes alpha:                           0.1
Beste Max Depth:                        2
Beste Max Features:                     3
Beste Max Estimators:                   500
MSE (eigne-Params):     2.0793059664380977
MSE (Paper-Params):     2.154301501002663
MSE (RF-Paper-Params):  2.039117028657757
MSE (RF-eigene-Params): 2.0925393121642575
MSE (Mean-Pred):        3.1220984006176984
MSE (Weighted Mean):    2.5878126720598154
MSE (NN):               2.7495785391862944

FS1+2: 
Bestes lambda (Regularisierung-Stärke): 0.3
Beste l1-ratio (bestes alpha):          0.7
Beste Größe Hidden Layer:               3
Bestes alpha:                           1.0
Beste Max Depth:                        2
Beste Max Features:                     3
Beste Max Estimators:                   500
MSE (eigne-Params):     2.186146005059289
MSE (Paper-Params):     2.4243

In [25]:
list_of_mse_nn

[2.7495785391862944,
 12.01772183106281,
 20.966318856070824,
 9.042000513040142,
 4.9876255575183475,
 12.245617827643844,
 15.507292681245167,
 8.414239957037209,
 5.99397790649591,
 13.267155703468495,
 21.10868966865947,
 6.398563442984851,
 4.916567234349062,
 12.044645380449204,
 16.14647089247592,
 7.262538255471832,
 7.272526970337758,
 11.554234694004059,
 13.655817229107146,
 8.986497417508883]

In [26]:
len(list_of_coefs[3])

315

In [27]:
list_of_coefs[3]

array([ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        1.12464171,  0.        ,  0.        ,  0.        , -0.        ,
       -0.        , -0.        , -0.        , -0.        , -0.        ,
       -0.        , -0.        , -0.        , -0.        , -0.        ,
        0.        ,  0.        , -0.        , -0.        ,  0.        ,
        0.        ,  0.        ,  0.        , -0.        , -0.        ,
       -0.        , -0.        ,  0.        ,  0.        ,  0.        ,
       -0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
       -0.        , -0.        , -0.        , -0.        ,  0.        ,
        0.        ,  0.        ,  0.        , -0.        , -0.        ,
        0.        , -0.        , -0.        , -0.        ,  0.        ,
        0.        , -0.        ,  0.        , -0.        , -0.        ,
        0.        ,  0.        ,  0.        ,  0.        , -0.        ,
        0.        ,  0.        ,  0.        , -0.        , -0.  

In [28]:
zuteilung = pd.read_excel('Zuteilung.xlsx')
zuteilung

,Kategorie,Code,Name
0,FS1,First,Lag
1,FS1,DNGDP2,NGDP Growth
2,FS1,DNGDP3,NGDP Growth
3,FS1,DNGDP4,NGDP Growth
4,FS1,DNGDP5,NGDP Growth
...,...,...,...
87,Investments,NOPIY,NaN
88,Profits,OIADPY,NaN
89,Profits,SALEY,NaN
90,Capital,RLLQ,NaN


In [32]:
index_list = {}
cnt = 0
for x in [fs1_wo_lag.columns[2:].values, fs1_lag.columns[2:3].values, fs2.columns[2:].values, fs3.columns[2:].values]:
    for y in x:
        index_list.update({cnt: f"{y}"})
        cnt+=1

index_list

{0: 'DNGDP2_mean',
 1: 'DNGDP3_mean',
 2: 'DNGDP4_mean',
 3: 'DNGDP5_mean',
 4: 'DNGDP6_mean',
 5: 'DNGDP2_median',
 6: 'DNGDP3_median',
 7: 'DNGDP4_median',
 8: 'DNGDP5_median',
 9: 'DNGDP6_median',
 10: 'RGDP2_mean',
 11: 'RGDP3_mean',
 12: 'RGDP4_mean',
 13: 'RGDP5_mean',
 14: 'RGDP6_mean',
 15: 'RGDP2_median',
 16: 'RGDP3_median',
 17: 'RGDP4_median',
 18: 'RGDP5_median',
 19: 'RGDP6_median',
 20: 'First',
 21: 'prccq_sd_dif',
 22: 'prccq_sd_yoy',
 23: 'prccq_vwmean_dif',
 24: 'prccq_mean_dif',
 25: 'prccq_vwmean_yoy',
 26: 'prccq_mean_yoy',
 27: 'prchq_sd_dif',
 28: 'prchq_sd_yoy',
 29: 'prchq_vwmean_dif',
 30: 'prchq_mean_dif',
 31: 'prchq_vwmean_yoy',
 32: 'prchq_mean_yoy',
 33: 'prclq_sd_dif',
 34: 'prclq_sd_yoy',
 35: 'prclq_vwmean_dif',
 36: 'prclq_mean_dif',
 37: 'prclq_vwmean_yoy',
 38: 'prclq_mean_yoy',
 39: 'apq_sd_dif',
 40: 'apq_sd_yoy',
 41: 'apq_vwmean_dif',
 42: 'apq_mean_dif',
 43: 'apq_vwmean_yoy',
 44: 'apq_mean_yoy',
 45: 'capsq_sd_dif',
 46: 'capsq_sd_yoy',
 47:

In [33]:
coef_df = pd.DataFrame(list_of_coefs[3])
coef_df

,0
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0
...,...
310,-0.0
311,0.0
312,0.0
313,0.0


In [34]:
coef_df.rename(index=index_list, inplace=True)
coef_df

,0
DNGDP2_mean,0.0
DNGDP3_mean,0.0
DNGDP4_mean,0.0
DNGDP5_mean,0.0
DNGDP6_mean,0.0
...,...
xsgaq_vwmean_dif,-0.0
xsgaq_mean_dif,0.0
xsgaq_vwmean_yoy,0.0
xsgaq_mean_yoy,0.0


In [35]:
grenzwert = 0.0001
coef_df.loc[np.abs(coef_df[coef_df.columns[0]]) > grenzwert]

,0
DNGDP2_median,1.124642
cstkq_mean_dif,-0.134365
dvpsxq_vwmean_dif,-0.137989
epsfiq_mean_yoy,0.005562
epspxq_mean_yoy,0.130429
epsx12_mean_dif,0.066754
loq_sd_yoy,0.025970
saleq_mean_dif,0.002108


In [36]:
feature_importance_list = {'Current Q'      : list_of_coefs[3],
                           'Current Q+1'    : list_of_coefs[7],
                           'Current Q+2'    : list_of_coefs[11],
                           'Current Q+3'    : list_of_coefs[15],
                           'Current Q+4'    : list_of_coefs[19]}
index_list = {}
cnt = 0
for x in [fs1_wo_lag.columns[2:].values, fs1_lag.columns[2:3].values, fs2.columns[2:].values, fs3.columns[2:].values]:
    for y in x:
        index_list.update({cnt: f"{y}"})
        cnt+=1
len(index_list)

314

In [37]:
importent_features = {}
for key, value in feature_importance_list.items():
    print(f"\033[1m{key}: \033[0m")
    coef_df = pd.DataFrame(value)
    coef_df.rename(index=index_list, inplace=True)
    grenzwert = 0.01
    print(len(coef_df.loc[np.abs(coef_df[coef_df.columns[0]]) > grenzwert]))
    print(coef_df.loc[np.abs(coef_df[coef_df.columns[0]]) > grenzwert])
    importent_features.update({key: coef_df.loc[np.abs(coef_df[coef_df.columns[0]]) > grenzwert]})



Current Q: 
6
                          0
DNGDP2_median      1.124642
cstkq_mean_dif    -0.134365
dvpsxq_vwmean_dif -0.137989
epspxq_mean_yoy    0.130429
epsx12_mean_dif    0.066754
loq_sd_yoy         0.025970
Current Q+1: 
138
                         0
DNGDP2_mean       0.018679
DNGDP3_mean       0.022096
DNGDP4_mean       0.012715
DNGDP2_median     0.021473
DNGDP3_median     0.022547
...                    ...
xoprq_mean_yoy   -0.017066
xsgaq_sd_dif     -0.010860
xsgaq_vwmean_dif  0.027406
xsgaq_mean_dif   -0.011406
xsgaq_vwmean_yoy -0.011356

[138 rows x 1 columns]
Current Q+2: 
1
                      0
csh12q_sd_yoy -0.034771
Current Q+3: 
5
                          0
csh12q_sd_yoy     -0.051601
csh12q_vwmean_dif -0.101884
lseq_sd_dif        0.062758
ppentq_vwmean_yoy -0.121685
txtq_vwmean_dif   -0.101977
Current Q+4: 
27
                          0
cheq_mean_dif      0.016315
csh12q_sd_yoy     -0.031239
csh12q_vwmean_dif -0.081968
dvpq_sd_yoy       -0.036571
epsfiq_vwmean_yoy -

In [38]:
for key, value in importent_features.items():
    print(f"\033[1m{key}: \033[0m")
    value['Kategorie'] = ""
    for s in value.index:
        f = s.split('_')
        if f[0].upper() != 'Q':
            if f[0].upper() in zuteilung['Code'].values:
                value.at[s, 'Kategorie'] = zuteilung.loc[zuteilung['Code'] == f[0].upper(), 'Kategorie'].values[0]
                # print(zuteilung.loc[zuteilung['Code'] == f.upper(), 'Kategorie'].values[0])
        else:
            print(f[0].upper())
            print(f[1].upper())
            if f[1].upper() in zuteilung['Code'].values:
                value.at[s, 'Kategorie'] = zuteilung.loc[zuteilung['Code'] == f[1].upper(), 'Kategorie'].values[0]

    print(value)

Current Q: 
                          0    Kategorie
DNGDP2_median      1.124642          FS1
cstkq_mean_dif    -0.134365  Investments
dvpsxq_vwmean_dif -0.137989      Profits
epspxq_mean_yoy    0.130429      Profits
epsx12_mean_dif    0.066754       Shares
loq_sd_yoy         0.025970    Liability
Current Q+1: 
Q
CSHPRY
Q
EPSFIY
Q
IBCOMY
Q
IBY
Q
NIY
Q
NIY
Q
NOPIY
Q
OIADPY
Q
OIADPY
Q
SALEY
                         0  Kategorie
DNGDP2_mean       0.018679        FS1
DNGDP3_mean       0.022096        FS1
DNGDP4_mean       0.012715        FS1
DNGDP2_median     0.021473        FS1
DNGDP3_median     0.022547        FS1
...                    ...        ...
xoprq_mean_yoy   -0.017066    Profits
xsgaq_sd_dif     -0.010860  Write-off
xsgaq_vwmean_dif  0.027406  Write-off
xsgaq_mean_dif   -0.011406  Write-off
xsgaq_vwmean_yoy -0.011356  Write-off

[138 rows x 2 columns]
Current Q+2: 
                      0    Kategorie
csh12q_sd_yoy -0.034771  Investments
Current Q+3: 
                          

In [41]:
importent_features['Current Q'].groupby(['Kategorie']).apply(lambda c: c.abs().sum())

,0
Kategorie,
FS1,1.124642
Investments,0.134365
Liability,0.025970
Profits,0.268418
Shares,0.066754


In [42]:
importent_features['Current Q+1'].groupby(['Kategorie']).apply(lambda c: c.abs().sum())

,0
Kategorie,
Capital,0.154472
FS1,0.107520
FS2,0.147372
Investments,0.275343
Liability,0.427721
Profits,0.676636
Shares,0.190693
Write-off,0.390532


In [43]:
importent_features['Current Q+2'].groupby(['Kategorie']).apply(lambda c: c.abs().sum())

,0
Kategorie,
Investments,0.034771


In [44]:
importent_features['Current Q+3'].groupby(['Kategorie']).apply(lambda c: c.abs().sum())

,0
Kategorie,
Capital,0.121685
Investments,0.153486
Liability,0.062758
Profits,0.101977


In [45]:
importent_features['Current Q+4'].groupby(['Kategorie']).apply(lambda c: c.abs().sum())

,0
Kategorie,
Capital,0.070024
Investments,0.131650
Liability,0.197268
Profits,0.392509
Shares,0.036571
Write-off,0.017880
